# Notebook 01 – Exploração de Dados e Limites de Métricas Univariadas

## Aula 4: Métricas Avançadas para Detecção de Drift

### Objetivos
- Gerar e explorar o dataset sintético de **crédito fintech** descrito no Documento 04
- Analisar as distribuições marginais das features entre períodos de referência e atual
- Aplicar testes univariados clássicos (**KS** e **PSI**) e verificar que **não detectam** o drift presente
- Revelar o **drift multivariado sutil** — inversão de correlação renda–idade — que escapa dessas métricas
- Motivar a necessidade de métricas avançadas (MMD, Wasserstein, Energy Distance)

### Teoria-Chave (Documento 04)

> **Definição de Drift:** Uma mudança nos dados ocorre quando a distribuição conjunta
> $P_{t_0}(X, y) \neq P_{t_1}(X, y)$, podendo se manifestar como *covariate shift*,
> *prior probability shift* ou *concept drift*.  
> — Gama et al. (2014)

> **Limitação de métricas univariadas:** Testes como KS e PSI avaliam cada variável
> isoladamente. Quando as marginais permanecem estáveis mas a **estrutura de correlação**
> se altera, esses testes produzem falsos negativos.  
> A **maldição da dimensionalidade** (Bellman, 1961; Aggarwal et al., 2001) agrava o
> problema: em alta dimensão, pontos se tornam equidistantes e os testes perdem poder.

### Vídeo Relacionado
**Vídeo 1 — Métricas Avançadas para Detecção de Drift** (≈15 min):  
Limites de testes univariados (KS, PSI) em alta dimensionalidade;  
maldição da dimensionalidade; drift multivariado sutil;  
caso fintech com drift não detectado por métodos clássicos.

In [ ]:
# Imports
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Adiciona diretório raiz da aula ao path
sys.path.insert(0, str(Path.cwd().parent))

from src.data_preprocessing import DataPreprocessor
from src.model import PSICalculator
from src.evaluation import plot_scatter_drift, plot_correlation_heatmaps

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Geração e Carregamento do Dataset Sintético

O dataset simula o cenário descrito no **Documento 04**: uma fintech de crédito
cujo modelo foi treinado em dados estáveis (período de referência). Meses depois,
a inadimplência aumenta sem que os testes univariados acusem mudança.

O segredo está na **estrutura de correlação** entre as variáveis: no período de
referência, renda e idade são positivamente correlacionadas (pessoas mais velhas
tendem a ter renda maior). No período atual, essa correlação se **inverte**,
configurando um drift multivariado sutil que afeta diretamente o risco de crédito.

Features geradas: `idade`, `renda`, `divida`, `score_credito`, `tempo_emprego`, `num_parcelas`.

In [ ]:
# Gera o dataset sintético
preprocessor = DataPreprocessor(n_samples=5000, seed=42)
df = preprocessor.generate_dataset()

print(f"Shape do dataset: {df.shape}")
print(f"\nColunas: {list(df.columns)}")
print()
df.info()
print()
print("Distribuição por período:")
print(df["periodo"].value_counts())
print()
df.head(10)

## 2. Análise Exploratória dos Dados (EDA)

Antes de aplicar qualquer teste de drift, vamos examinar as estatísticas descritivas
de cada período separadamente. Conforme o **Documento 04**, as **distribuições marginais**
são propositalmente similares entre os períodos — é exatamente isso que torna o drift
"sutil" e difícil de detectar com métricas univariadas.

In [ ]:
# Separa os dados por período
df_ref, df_cur = preprocessor.split_data(df)

numeric_cols = ["idade", "renda", "divida", "score_credito", "tempo_emprego", "num_parcelas"]

print("=" * 70)
print("ESTATÍSTICAS DESCRITIVAS — PERÍODO DE REFERÊNCIA")
print("=" * 70)
print(df_ref[numeric_cols].describe().round(2).to_string())

print()
print("=" * 70)
print("ESTATÍSTICAS DESCRITIVAS — PERÍODO ATUAL")
print("=" * 70)
print(df_cur[numeric_cols].describe().round(2).to_string())

print("\n→ Observe como médias e desvios-padrão são muito similares entre períodos.")

## 3. Distribuições Marginais

Conforme discutido no **Documento 04** (Vídeo 1), testes como **KS** e **PSI** avaliam
cada feature **isoladamente** — ou seja, comparam distribuições marginais univariadas.

Vamos visualizar os histogramas de cada feature nos dois períodos. A hipótese é que
as distribuições marginais serão visualmente **muito similares**, mesmo havendo drift
na estrutura de correlação.

In [ ]:
# Histogramas comparativos: referência vs atual para cada feature
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    ax = axes[i]
    ax.hist(df_ref[col], bins=40, alpha=0.5, density=True,
            label="Referência", color="steelblue")
    ax.hist(df_cur[col], bins=40, alpha=0.5, density=True,
            label="Atual", color="tomato")
    ax.set_title(col, fontsize=12)
    ax.set_ylabel("Densidade")
    ax.legend(fontsize=9)

fig.suptitle("Distribuições Marginais: Referência vs Atual\n"
             "(Documento 04 — as marginais parecem similares)",
             fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

print("→ Visualmente, as distribuições marginais são praticamente idênticas.")
print("  Isso explica por que métricas univariadas falham em detectar o drift.")

## 4. Teste KS Univariado

O teste de **Kolmogorov-Smirnov (KS)** compara duas distribuições empíricas medindo
a maior diferença absoluta entre suas funções de distribuição acumulada (CDFs).

Segundo o **Documento 04**, este teste avalia cada variável **isoladamente** e, portanto,
não captura mudanças na **estrutura de dependência** entre variáveis. Em cenários de
alta dimensionalidade, a **maldição da dimensionalidade** (Bellman, 1961) reduz ainda
mais seu poder estatístico.

In [ ]:
# Teste KS univariado para cada feature
print("TESTE KS UNIVARIADO — Feature a Feature")
print("=" * 60)
print(f"{'Feature':<18} {'Estatística KS':>15} {'p-valor':>12} {'Drift?':>10}")
print("-" * 60)

alpha = 0.05
for col in numeric_cols:
    stat, pvalue = stats.ks_2samp(df_ref[col].values, df_cur[col].values)
    drift = "SIM" if pvalue < alpha else "NÃO"
    print(f"{col:<18} {stat:>15.4f} {pvalue:>12.4f} {drift:>10}")

print("-" * 60)
print(f"\nNível de significância α = {alpha}")
print("→ O teste KS NÃO detecta drift significativo nas marginais.")
print("  Isso é um FALSO NEGATIVO: o drift existe, mas é multivariado.")

## 5. PSI Univariado

O **Population Stability Index (PSI)** é outra métrica univariada amplamente utilizada
na indústria financeira para monitorar estabilidade de modelos.

Conforme o **Documento 04** (Snippet 1 — Hands On), a fórmula do PSI é:

$$\text{PSI} = \sum_{i=1}^{B} (p_i - q_i) \cdot \ln\left(\frac{p_i}{q_i}\right)$$

onde $p_i$ e $q_i$ são as proporções em cada bin nos períodos de referência e atual.

**Limiares usuais:**
- PSI < 0,10 → Sem mudança significativa
- 0,10 ≤ PSI < 0,25 → Mudança moderada
- PSI ≥ 0,25 → Drift significativo

Assim como o KS, o PSI analisa cada variável **isoladamente** (Documento 04, Seção
"Limites de Métricas Simples").

In [ ]:
# PSI univariado por feature usando PSICalculator do src.model
psi_calc = PSICalculator(n_bins=10)

ref_array = df_ref[numeric_cols].values
cur_array = df_cur[numeric_cols].values

psi_values = psi_calc.calculate_multifeature(ref_array, cur_array)

print("PSI UNIVARIADO — Feature a Feature")
print("=" * 55)
print(f"{'Feature':<18} {'PSI':>10} {'Interpretação':>22}")
print("-" * 55)

for col, psi in zip(numeric_cols, psi_values):
    if psi >= 0.25:
        interp = "⚠ Drift significativo"
    elif psi >= 0.10:
        interp = "~ Mudança moderada"
    else:
        interp = "✓ Sem mudança"
    print(f"{col:<18} {psi:>10.4f} {interp:>22}")

print("-" * 55)
print(f"\nLimiar de drift significativo: PSI ≥ 0.25 (Documento 04)")
print("→ Nenhuma feature ultrapassa o limiar: PSI indica 'sem drift'.")
print("  Novamente, um FALSO NEGATIVO — o drift multivariado não é capturado.")

## 6. O Ponto Cego: Drift Multivariado Sutil

Conforme demonstrado no **Documento 04**, os testes KS e PSI falharam porque avaliam
cada variável **marginalmente**. No entanto, a **distribuição conjunta** $P(X)$ mudou:
a correlação entre `idade` e `renda` **se inverteu** do período de referência para o atual.

Esse é exatamente o tipo de drift que motiva o uso de **métricas multivariadas** como:
- **MMD** (Maximum Mean Discrepancy) — Gretton et al. (2012)
- **Wasserstein Distance** — distância de transporte ótimo
- **Energy Distance** — Székely & Rizzo (2013)

Vamos visualizar a inversão de correlação.

In [ ]:
# Scatter plot: idade vs renda — revelando a inversão de correlação
# Usa plot_scatter_drift do src.evaluation (Documento 04, Figura 1)
fig = plot_scatter_drift(
    reference=ref_array,
    current=cur_array,
    feature_x=0,  # idade
    feature_y=1,  # renda
    labels=("Idade", "Renda"),
)
plt.show()

print("→ O scatter plot revela claramente a INVERSÃO da correlação idade-renda.")
print("  Referência (azul): correlação POSITIVA — idade ↑ → renda ↑")
print("  Atual (vermelho): correlação NEGATIVA — idade ↑ → renda ↓")

In [ ]:
# Heatmaps de correlação lado a lado
# Usa plot_correlation_heatmaps do src.evaluation (Documento 04, Vídeo 1)
fig = plot_correlation_heatmaps(
    reference=ref_array,
    current=cur_array,
    feature_names=numeric_cols,
)
plt.show()

print("→ Compare as células (idade, renda) nos dois heatmaps.")
print("  A correlação mudou de sinal — evidência de drift multivariado.")

## 7. Por que métricas univariadas falham?

O **Documento 04** identifica dois fatores principais:

1. **Suposição de independência:** KS e PSI tratam cada feature de forma isolada.
   Se as marginais $P(X_i)$ permanecem estáveis, mas a distribuição conjunta
   $P(X_1, X_2, \ldots, X_d)$ muda (por exemplo, por inversão de correlação),
   esses testes **não detectam a mudança**.

2. **Maldição da dimensionalidade** (Bellman, 1961; Aggarwal et al., 2001):
   Em espaços de alta dimensão, a noção de distância se degrada — todos os pontos
   se tornam aproximadamente equidistantes. Isso reduz o poder de qualquer teste
   que dependa de particionamento do espaço (como bins do PSI).

A solução: utilizar métricas que operam sobre a **distribuição conjunta** completa,
como o **MMD**, que mapeia distribuições para um espaço de Hilbert via kernel:

$$\text{MMD}^2(P, Q) = \mathbb{E}[k(x,x')] + \mathbb{E}[k(y,y')] - 2\mathbb{E}[k(x,y)]$$

— Gretton et al. (2012), conforme Documento 04, Snippet 2.

In [ ]:
# Evidência quantitativa: correlação de Pearson para pares-chave
print("CORRELAÇÃO DE PEARSON — idade vs renda")
print("=" * 50)

corr_ref = np.corrcoef(df_ref["idade"].values, df_ref["renda"].values)[0, 1]
corr_cur = np.corrcoef(df_cur["idade"].values, df_cur["renda"].values)[0, 1]

print(f"Período de Referência:  r = {corr_ref:+.4f}")
print(f"Período Atual:          r = {corr_cur:+.4f}")
print(f"Diferença absoluta:     Δr = {abs(corr_ref - corr_cur):.4f}")
print()

if np.sign(corr_ref) != np.sign(corr_cur):
    print("⚠ INVERSÃO DE SINAL detectada!")
    print("  A correlação passou de POSITIVA para NEGATIVA (ou vice-versa).")
    print("  Isso configura drift multivariado sutil — exatamente o cenário")
    print("  descrito no Documento 04.")
else:
    print("  Correlação manteve o mesmo sinal.")

print()
print("Correlação completa — Referência:")
print(df_ref[numeric_cols].corr().round(3).to_string())
print()
print("Correlação completa — Atual:")
print(df_cur[numeric_cols].corr().round(3).to_string())

## Resumo

### O que aprendemos neste notebook

1. **Distribuições marginais similares** não garantem ausência de drift — a distribuição
   conjunta $P(X)$ pode ter mudado significativamente.

2. **Teste KS** (Kolmogorov-Smirnov): não detectou drift em nenhuma feature individual,
   pois as marginais são de fato estáveis.

3. **PSI** (Population Stability Index): todos os valores ficaram abaixo do limiar de 0,25,
   indicando falsamente "sem drift".

4. **O drift real** está na **inversão da correlação** entre `idade` e `renda` — um
   fenômeno **multivariado** que testes univariados são incapazes de capturar.

5. A **maldição da dimensionalidade** (Bellman, 1961) agrava a limitação dessas métricas
   em espaços de alta dimensão.

### Próximo Notebook

No **Notebook 02 — Métricas Multivariadas**, aplicaremos métricas que operam sobre a
distribuição conjunta completa:
- **MMD** (Maximum Mean Discrepancy) — Gretton et al. (2012)
- **Wasserstein Distance** — distância de transporte ótimo
- **Energy Distance** — Székely & Rizzo (2013)

Essas métricas, discutidas no **Documento 04**, são projetadas para capturar exatamente
o tipo de drift sutil que demonstramos aqui.